<a href="https://colab.research.google.com/github/theo-lynn/ML-Musical-Instrument-Harmnonic-Recognition-/blob/main/instrument_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Musical Instrument Recognition from Spectrograms

Each instrument has a characteristic pattern of harmonics (its *timbre*). A mel spectrogram turns
those harmonics into an image — frequency on the y-axis, time on the x-axis — so we can classify
instruments with an image model.

**Pipeline**
1. Download the [Musical Instruments Sound Dataset](https://www.kaggle.com/datasets/soumendraprasad/musical-instruments-sound-dataset) (guitar, drums, violin, piano)
2. Check the data for duplicate / conflicting files
3. Convert every clip to a fixed-length mel spectrogram image
4. Fine-tune a pretrained ResNet34 with fastai
5. Evaluate on the held-out test set and export the model

Run on Colab with a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

## 1. Setup

Colab already has `fastai`, `librosa` and the `kaggle` CLI installed.

To download from Kaggle, add two Colab secrets (🔑 icon in the left sidebar): `KAGGLE_USERNAME` and
`KAGGLE_KEY` — both come from *kaggle.com → Settings → API → Create New Token*.
Outside Colab, put `kaggle.json` in `~/.kaggle/` instead.

In [ ]:
import os
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from PIL import Image
from fastai.vision.all import *

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except ImportError:
    pass  # not on Colab: kaggle reads ~/.kaggle/kaggle.json

In [ ]:
DATA_DIR = Path('data')
SPEC_DIR = Path('spectrograms')

# Kaggle label -> folder / class name
CLASSES = {
    'Sound_Guitar': 'guitar',
    'Sound_Drum': 'drums',
    'Sound_Violin': 'violin',
    'Sound_Piano': 'piano',
}

SAMPLE_RATE = 22050
DURATION = 3.0     # seconds; every clip is trimmed or zero-padded to this length
N_MELS = 128
HOP_LENGTH = 512
IMG_SIZE = 224

## 2. Download the data

In [ ]:
if not (DATA_DIR / 'Metadata_Train.csv').exists():
    !kaggle datasets download -d soumendraprasad/musical-instruments-sound-dataset -p {DATA_DIR}
    !unzip -q -n {DATA_DIR}/musical-instruments-sound-dataset.zip -d {DATA_DIR}
    !rm {DATA_DIR}/musical-instruments-sound-dataset.zip

TRAIN_AUDIO = DATA_DIR / 'Train_submission' / 'Train_submission'
TEST_AUDIO = DATA_DIR / 'Test_submission' / 'Test_submission'

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'Metadata_Train.csv')
test_df = pd.read_csv(DATA_DIR / 'Metadata_Test.csv')
train_df['path'] = train_df['FileName'].map(lambda f: TRAIN_AUDIO / f)
test_df['path'] = test_df['FileName'].map(lambda f: TEST_AUDIO / f)

print('Train:'); print(train_df['Class'].value_counts(), '\n')
print('Test:'); print(test_df['Class'].value_counts())

## 3. Check the data

Datasets scraped from the web often contain the same recording more than once. Two problems to look for:
- **Label conflicts** — the same audio filed under two different instruments. These teach the model contradictory things.
- **Train/test leakage** — the same audio in both splits, which inflates the test score.

We hash the raw audio bytes to find exact duplicates.

In [ ]:
def file_hash(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

for df in (train_df, test_df):
    df['exists'] = df['path'].map(lambda p: p.exists())
    print(f"missing files: {(~df['exists']).sum()}")
train_df = train_df[train_df['exists']].copy()
test_df = test_df[test_df['exists']].copy()

train_df['hash'] = train_df['path'].map(file_hash)
test_df['hash'] = test_df['path'].map(file_hash)

labels_per_hash = train_df.groupby('hash')['Class'].nunique()
conflicting = labels_per_hash[labels_per_hash > 1].index
print(f'train clips with conflicting labels: {train_df["hash"].isin(conflicting).sum()}')
print(f'exact duplicate clips in train:     {train_df.duplicated("hash").sum()}')
print(f'test clips also in train (leakage): {test_df["hash"].isin(train_df["hash"]).sum()}')

In [ ]:
# Drop conflicting clips, keep one copy of each duplicate, and remove leaked test clips
train_df = train_df[~train_df['hash'].isin(conflicting)].drop_duplicates('hash')
test_df = test_df[~test_df['hash'].isin(train_df['hash'])].drop_duplicates('hash')

print('Clean train:'); print(train_df['Class'].value_counts(), '\n')
print('Clean test:'); print(test_df['Class'].value_counts())

## 4. Audio → mel spectrogram images

Every clip is trimmed or padded to exactly `DURATION` seconds, so one pixel always covers the same slice
of time and frequency. Without this, short clips get stretched to fill the image and their harmonics end
up looking different from long clips of the same instrument.

The spectrogram is saved directly as a grayscale image (low frequencies at the bottom) — no matplotlib
figure, so there are no axes, borders or colormap artefacts, and it's much faster.

In [ ]:
def audio_to_mel_image(audio_path):
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True, duration=DURATION)
    y = librosa.util.fix_length(y, size=int(SAMPLE_RATE * DURATION))
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)  # range [-80, 0] dB
    pixels = ((mel_db + 80.0) / 80.0 * 255).astype(np.uint8)
    return Image.fromarray(np.flipud(pixels))


def build_spectrograms(df, split):
    failed = []
    for row in progress_bar(list(df.itertuples())):
        dest = SPEC_DIR / split / CLASSES[row.Class] / f'{Path(row.FileName).stem}.png'
        if dest.exists():
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            audio_to_mel_image(row.path).save(dest)
        except Exception as e:
            failed.append((row.FileName, repr(e)))
    print(f'{split}: {len(df) - len(failed)} images, {len(failed)} failed')
    return failed


train_failed = build_spectrograms(train_df[train_df['Class'].isin(CLASSES)], 'train')
test_failed = build_spectrograms(test_df[test_df['Class'].isin(CLASSES)], 'test')

In [ ]:
# One example per instrument
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4 * len(CLASSES), 3))
for ax, name in zip(axes, CLASSES.values()):
    img = Image.open(next((SPEC_DIR / 'train' / name).iterdir()))
    ax.imshow(img, cmap='magma', aspect='auto')
    ax.set_title(name)
    ax.axis('off')
plt.show()

## 5. Train

The usual image augmentations (flips, rotations, zooms) are deliberately **not** used: flipping a
spectrogram vertically swaps high and low frequencies, which would destroy the harmonic pattern we're
trying to learn.

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(IMG_SIZE, method=ResizeMethod.Squish),
).dataloaders(SPEC_DIR / 'train', bs=32)

print(dls.vocab)
dls.show_batch(max_n=8, figsize=(10, 5))

In [ ]:
learn = vision_learner(dls, resnet34, metrics=[accuracy, error_rate])
lr = learn.lr_find().valley

In [ ]:
learn.fine_tune(8, base_lr=lr, freeze_epochs=1, cbs=SaveModelCallback(monitor='valid_loss'))
learn.recorder.plot_loss()

## 6. Evaluate

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(5, 5))
interp.plot_top_losses(6, nrows=2)

Validation images come from the same pool as training. The **test set** below is a separate set of recordings the model has never seen, so it's the honest score.

In [ ]:
from sklearn.metrics import classification_report

test_files = get_image_files(SPEC_DIR / 'test')
test_dl = dls.test_dl(test_files, with_labels=True)
probs, targets = learn.get_preds(dl=test_dl)
preds = probs.argmax(dim=1)

print(f'Test accuracy: {(preds == targets).float().mean():.3f}  ({len(test_files)} clips)\n')
print(classification_report(targets, preds, labels=range(len(dls.vocab)), target_names=dls.vocab, zero_division=0))

## 7. Export and predict on new audio

In [ ]:
learn.export('instrument_classifier.pkl')

In [ ]:
def predict_instrument(audio_path, learner=learn):
    img = PILImage.create(audio_to_mel_image(audio_path).convert('RGB'))
    label, _, probs = learner.predict(img)
    return label, dict(zip(learner.dls.vocab, probs.numpy().round(3)))

predict_instrument(test_df['path'].iloc[0])